<div style="background-color: darkslategray; padding: 10px; border-radius: 5px">

## Be stars scheduling

### This code uses a list of all classical Be stars from BeSS (http://basebe.obspm.fr/basebe/BeSS/MenuStarConsulReq.php) <br>with 0 < Vmag < 8, and then allows the user to implement cuts on this catalog based on Vmag, RA (hours), and Dec (degrees).

### Various outputs and an RLMT scheduling file are created.

<!-- <span style="color: orange;">## Various outputs and an RLMT scheduling file are created.</span> -->

### MPT25
</div>

# First a block for the user to define important selection and observing parameters

In [ ]:
import os

os.path.isdir('data/2025-10-25')

In [ ]:
# Selection criteria to extract targets from the BeSS database:
Vmaglow = 0
Vmaghigh = 8.0
RAlow  = 22 # hours
RAhigh = 6 # hours
Declow = -10 # deg
Dechigh = 72 # deg

# Exposure time calculator values
seeing_fwhm = 2.4 # may want to modify; median value for Winer is 2.4".
exptime = 3. # value here doesn't matter; ETC estimates stauration time independently of this value
filter = "g" # can only be Sloan g', r', i', or ha.  None are perfect matches to Vmag for Be stars from BeSS.  Use g'
magnitude = 8.0 # value here doesn't matter; changed in code below using Vmag from BeSS
telescope = "RLMT" # leave as is
camera = "ZWO ASI6200MM" # leave as is

# List of filters to use in ETC calculation:
filter_list = ['g', 'r', 'i', 'ha']
filter_list = ['r', 'ha']

# Organization:

In [ ]:
from astropy import time
import os,sys
from datetime import datetime,timedelta


# Get current date in ISO format
current_date = datetime.now().date()

# Folder name
folder_name = os.getcwd() + '/BeSS_Scheduling/' +str(current_date)+'/'

# Create the folder
if not os.path.exists(folder_name):
    os.makedirs(folder_name)
    print(f"Folder '{folder_name}' created successfully")
else:
    print(f"Folder '{folder_name}' already exists")

# file_path = "/content/drive/MyDrive/SP25-PHYS491/scheduling/"+str(date) # This is where you want to read/write from/to
# print('  ---> Working in '+str(file_path))

## Import packages and define functions; manually install a few packages (as necessary)

In [ ]:
# !pip install reproject
# !pip install astroquery
from astropy.io import fits
import glob
import shutil
import skimage.io
from astropy.wcs import WCS
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
import numpy as np
from astropy.coordinates import SkyCoord
from astroquery.simbad import Simbad
from astroquery.vizier import Vizier
from astropy.time import Time
import astropy.units as u
from astropy.visualization import astropy_mpl_style
from astropy.modeling.functional_models import Gaussian2D
from astropy.coordinates import Galactic
from astropy.coordinates import (
    Angle,
    SkyCoord,
    EarthLocation,
   #get_moon,
    solar_system_ephemeris,
    get_body,
)
from matplotlib.patches import Ellipse
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import astropy.units as u
import warnings
from pytz import timezone
from time import strftime
import pandas as pd
from IPython.display import display, HTML
warnings.filterwarnings('ignore')
# import beautifulsoup:
from bs4 import BeautifulSoup

# Function to convert RA to decimal degrees
def ra_to_decimal_degrees(ra):
    ra_parts = ra.split()
    hours = float(ra_parts[0])
    minutes = float(ra_parts[1])
    seconds = float(ra_parts[2])
    return 15.*(hours + minutes / 60 + seconds / 3600)

# Function to convert Dec to decimal degrees
def dec_to_decimal_degrees(dec):
    dec_parts = dec.split()
    degrees = float(dec_parts[0])
    minutes = float(dec_parts[1])
    seconds = float(dec_parts[2])
    if degrees < 0:
        return degrees - minutes / 60 - seconds / 3600
    else:
        return degrees + minutes / 60 + seconds / 3600

# Define custom CSS styles for the table
css_style = """
<style>
    table {
        background-color: gray;
        border: 2px solid black;
        width: 100%;
        text-align: center;
        color: #FFE4C4;
    }
    th, td {
        padding: 8px;
        border: 1px solid black;
    }
    th {
        font-weight: bold;
    }
    a {
        color: bisque;
    }
</style>
"""

###################################################
######### ETC from Philip Griffin follows #########
###################################################

micron = 1e-6


class Exptime_Calculator:
    def __init__(self, telescope, camera, object_name, filter, fwhm, time, magnitude):
        self.telescope = telescope
        self.camera = camera
        self.filter = filter
        self.fwhm = fwhm/206265.  # Seeing FWHM in arcsec
        self.time = time
        self.object_name = object_name
        self.magnitude = magnitude

        if telescope == "RLMT":
            longitude = "-110d36m06.42s"
            latitude = "+31d39m56.08s"
            elevation = 1500 * u.m
            location = EarthLocation.from_geodetic(
                longitude, latitude, elevation
            )
            obsname = "Robert L. Mutel Telescope (Winer Observatory)"
            # observer = Observer(name = obsname, location=location, timezone=timezone('US/Arizona'))
            description = "Robert L. Mutel Telescope at Winer Observatory"
            obscode = "857"
            self.obscode = obscode
            self.night_adu = 0.2  # Pseudo ADU/pixel/sec in G or R filter; night sky, no Moon
            self.min_elevation = 20 * u.degree
            D = 0.51
            f = 6.8
            self.plate_scale = D * f

        self.obsname = obsname
        # self.observer = observer
        self.obscode = obscode
        self.description = description

        # Camera parameters
        if "4040" in camera:
            self.RN = 2.7
            self.DC = 0.07  # Scaling from 0.3 at 0C to -20C
            self.G_high = (
                1.05  # (inverse) high gain, elec/ADU unbinned (2.3 binned)
            )
            self.G_low = (
                1.05 / 16.0
            )  # (inverse) low gain, elec/ADU unbinned (2.3 binned)
            self.pixel_size = 9 * micron  # unbinned
            self.camera_gain_modes = ["Low", "High", "Spro"]
            self.saturation_adu = {"Low": 3000, "High": 3000, "Spro": 55000}
            self.gains = {"Low": 1 / 16, "High": 1.05, "Spro": 1.05}

        elif "6303" in camera:
            self.RN = 15
            self.DC = 0.04  # Scaling from 0.3 at 0C to -20C
            self.G_high = (
                1.05  # (inverse) high gain, elec/ADU unbinned (2.3 binned)
            )
            self.G_low = (
                1.05 / 16.0
            )  # (inverse) low gain, elec/ADU unbinned (2.3 binned)
            self.pixel_size = 9 * micron  # unbinned
            self.camera_gain_modes = ["Default"]
            self.saturation_adu = {"Default": 55000}
            self.gains = {"Default": 1}  # True??
        elif "6200" in camera:
            self.RN = 1.5
            self.DC = 1e-2
            self.G_high = 1.0  # (inverse) high gain, elec/ADU unbinned
            self.G_low = 1.0
            self.night_adu = 0.4
            self.pixel_size = 7.52 * micron
            self.camera_gain_modes = ["Default"]
            self.saturation_adu = {"Default": 55000}
            self.gains = {"Default": 1}

            if telescope == "RLMT":
                self.ZPmag = {
                    "g": 22.6,
                    "r": 21.8,
                    "i": 20.7,
                    "ha": 18.9,  # Based on HD62367 on 2024-12-26
                }  # Initial estimates with ZWO 6200

    def set_obj_name(self, object_name):
        self.object_name = object_name

    def set_magnitude(self, magnitude):
        self.magnitude = magnitude

    def createGaussian(amp, stddev, x_center=0, y_center=0):
        gauss = Gaussian2D(
            amplitude=amp,
            x_stddev=stddev,
            y_stddev=stddev,
            x_mean=x_center,
            y_mean=y_center,
        )

        intensity = 2 * np.pi * amp * stddev**2

        return gauss, intensity

    def calc_peak(self, intensity, fwhm_pixel):
        """Calculate the peak intensity of a gaussian

        Parameters
        ----------
        intensity:
            intensity of the entire gaussian

        fwhm_pixel:
            FWHM of the source in pixels

        Returns:
        --------
        amp:
            amplitude, or peak intensity
        """
        # First convert to Standard Deviation from FWHM
        stddev = fwhm_pixel / 2.355

        # Then calculate the amplitude using the intensity
        # formula from integrating a gaussian.
        # Intensity = 2*pi*Amp*sigma_x*sigma_y
        amp = intensity / (2 * np.pi * stddev**2)

        return amp

    def sky_params(self, mode, exptime):
        """Sky and source brightness and RMS in ADU"""
        filter = self.filter.lower()
        saturation_adu = self.saturation_adu[mode]
        fwhm_pixel = self.fwhm * self.plate_scale / self.pixel_size
        npixel = (
            1.75 * np.pi * (fwhm_pixel / 2) ** 2
        )  # This 1.75 must be some fit to the data for some old camera and may not reflect reality...
        RN = self.RN

        if mode == "Default":
            gain = self.G_high

        # factor to account for filter width - m/l guesses
        fw = {"g": 1.0, "r": 1.0, "i": 1.0, "ha": 0.04}
        night_sky = gain * fw[filter] * self.night_adu * exptime
        sky_mean = gain * (night_sky + (RN))
        sky_rms = np.sqrt(night_sky + RN**2)

        adu_sec = gain * 2.5 ** (self.ZPmag[filter] - self.magnitude)
        adu_sum = adu_sec * exptime
        adu_peak = self.calc_peak(adu_sum, fwhm_pixel)
        snr = adu_sum / np.sqrt(sky_rms + adu_sum * gain)
        saturation_time = saturation_adu / adu_peak * exptime
        return (
            sky_mean,
            sky_rms,
            adu_sec,
            adu_sum,
            snr,
            adu_peak,
            saturation_time,
        )

<div style="background-color: darkslategray; padding: 10px; border-radius: 5px">

### Read csv file into pandas dataframe

#### Add columns with RA and Dec converted to decimal degrees.
</div>

In [ ]:
# User input number of rows to display
nrows=10

csv_folder_name = os.getcwd() + '/BeSS_Scheduling/'

# Define the URL and file name
# url = "https://jcannon.macalester.digital/MACRO/PHYS491-SP25/BeSS.Classical.Vmag_0_8.csv"
file_name = csv_folder_name+"BeSS.Classical.Vmag_0_8.csv"

# Download the CSV file
# response = requests.get(url)
# response.raise_for_status()  # Check for any request errors

# # Save the file locally
# with open(file_name, "wb") as file:
#     file.write(response.content)

# print(f"File downloaded and saved as '{file_name}'.")

# Open the CSV file in a sortable format using pandas
df = pd.read_csv(file_name)
print('Total number of Be stars:', len(df))

# Apply the conversion functions
df['RA_deg'] = df['RA'].apply(ra_to_decimal_degrees)
df['Dec_deg'] = df['Dec'].apply(dec_to_decimal_degrees)

# Display the sorted DataFrame
# with pd.option_context('display.max_columns', None, 'display.max_rows', nrows, 'display.width', 1000):
#     columns_to_print = ['Name', 'RA', 'Dec', 'Vmag', 'SpType', 'Vsini', 'Nspectra', 'Comments', 'RA_deg', 'Dec_deg']
#     print(df[columns_to_print].to_string(na_rep=''))


<div style="background-color: darkslategray; padding: 10px; border-radius: 5px">

### User inputs ranges of Vmag, RA, DEC on which to select tatgets

### Display table and figure with resulting Be stars:

</div>

In [ ]:
# Filter data:
# filtered_df = df[(df['Vmag'] >= Vmaglow) & (df['Vmag'] <= Vmaghigh) & (df['RA_deg'] >= RAlow*15.) & (df['RA_deg'] <= RAhigh*15.) & (df['Dec_deg'] >= Declow) & (df['Dec_deg'] <= Dechigh)]

if RAlow < RAhigh:
    ra_mask = (df['RA_deg'] >= RAlow*15.) & (df['RA_deg'] <= RAhigh*15.)
else:
    # wrap-around case: e.g., 21h → 5h
    ra_mask = (df['RA_deg'] >= RAlow*15.) | (df['RA_deg'] <= RAhigh*15.)

filtered_df = df[
    (df['Vmag'] >= Vmaglow) & (df['Vmag'] <= Vmaghigh) &
    ra_mask &
    (df['Dec_deg'] >= Declow) & (df['Dec_deg'] <= Dechigh)
]

# print statement to show the number of selected stars:
print(f"Number of stars selected: {len(filtered_df)}")
print('   Selection criteria:')
print(f'      Vmag between {Vmaglow} and {Vmaghigh}')
print(f'      RA between {RAlow} and {RAhigh} hours')
print(f'      Dec between {Declow} and {Dechigh} degrees')

# Use Astropy's matplotlib style
plt.style.use(astropy_mpl_style)

# Plot configuration
fig, ax = plt.subplots(figsize=(10, 6))

# Plot the stars
ax.scatter(df['RA_deg'], df['Dec_deg'], color='gray', label=r'0<V$_{\rm mag}$<8 (n='+str(len(df))+')', s=6)
ax.scatter(filtered_df['RA_deg'], filtered_df['Dec_deg'], color='blue', label='Selected (n='+str(len(filtered_df))+')', s=6)

# Generate points for the Milky Way Galactic Plane
gal_l = np.linspace(0, 360, 1000)  # Galactic longitude
gal_b = np.zeros_like(gal_l)  # Galactic latitude (b=0 for the plane)

# Convert Galactic Plane coordinates to RA and Dec
galactic_plane_coords = SkyCoord(l=gal_l*u.degree, b=gal_b*u.degree, frame=Galactic)
ra_dec_coords = galactic_plane_coords.icrs
ra_plane = ra_dec_coords.ra.deg
dec_plane = ra_dec_coords.dec.deg

# Plot the Galactic Plane
ax.scatter(ra_plane, dec_plane, color='lightcoral', label='Galactic Plane',s=6)

# Label axes
ax.set_xlabel('Right Ascension (degrees)')
ax.set_ylabel('Declination (degrees)')

# limit the x-axis to the RA range of the selected Be stars:
# ax.set_xlim(RAlow*15., RAhigh*15.)

# limit the x-axis limits to be exactly 0 to 360:
ax.set_xlim(0., 360.)

# Add grid, legend, and title
ax.grid(True, which='both', linestyle='--', linewidth=0.5)
# ax.legend(facecolor='white', edgecolor='black')
ax.legend(facecolor='white', edgecolor='black', loc='center left', bbox_to_anchor=(1, 0.4))
ax.set_title(r'Classical Be Stars from BeSS')

# Invert RA axis for proper astronomical convention
ax.invert_xaxis()

# Add a text box with the Vmag, RA, and DEC ranges
text_box_content = (f"Selection critera:\n"
                    f"{Vmaglow} < V < {Vmaghigh}\n"
                    f"{RAlow*15:.1f}° ≤ RA < {RAhigh*15:.1f}°\n"
                    f"{Declow:.1f}° ≤ Dec < {Dechigh:.1f}°")

# ax.text(0.02, 0.98, text_box_content,
#         transform=ax.transAxes,
#         fontsize=12,
#         verticalalignment='top',
#         bbox=dict(facecolor='wheat', edgecolor='black', boxstyle='round,pad=0.5',alpha=0.6),
#         color='black')

fig.text(0.92, 0.6, text_box_content,
         fontsize=12,
         verticalalignment='center',
         bbox=dict(facecolor='wheat', edgecolor='black', boxstyle='round,pad=0.5', alpha=0.6),
         color='black')

# Show the plot
plt.show()
# save the plot:
fig.savefig(str(folder_name)+'BeSS.'+str( Time.now().iso.split(' ')[0])+'.png', dpi=300, bbox_inches='tight')

# Create the URL columns
filtered_df['Simbad_URL'] = filtered_df['Name'].apply(lambda star_name: 'https://simbad.u-strasbg.fr/simbad/sim-basic?Ident=' + str(star_name.replace(" ", "%20")))
filtered_df['BeSS_URL'] = filtered_df['Name'].apply(lambda star_name: 'http://basebe.obspm.fr/basebe/BeSS/Consul.php?specobj='+str(star_name.replace(" ", "%20")))

# Display the DataFrame with the added URL columns
with pd.option_context('display.max_columns', None, 'display.max_rows', None, 'display.width', 1000):
    columns_to_print = ['Name', 'RA', 'Dec', 'Vmag', 'SpType', 'Vsini', 'Nspectra', 'Comments', 'Simbad_URL', 'BeSS_URL']
    print(filtered_df[columns_to_print].to_string(na_rep=''))



<div style="background-color: darkslategray; padding: 10px; border-radius: 5px">

### Get the most recent spectrum from BeSS

</div>

In [ ]:
# add a new column to the dataframe to store the spectrum URL:
filtered_df['Output_URL'] = ''

# URL of the page to scrape

for j in range(len(filtered_df)):

    if filtered_df['Nspectra'].iloc[j] == 0:
        continue
    if filtered_df['Nspectra'].iloc[j] > 0:
        url = filtered_df['BeSS_URL'].iloc[j]
    name = filtered_df['Name'].iloc[j]

    # Send a GET request to the page
    response = requests.get(url)
    response.raise_for_status()  # Check for request errors

    # Parse the page content
    soup = BeautifulSoup(response.content, 'html.parser')

    # Base URL to search for in href attributes
    base_url = 'SendPng.php?v_ids='

    # Find all 'a' tags with href attributes containing the base URL
    links = soup.find_all('a', href=lambda href: href and base_url in href)

    numberlist = []
    for link in links:
        string = str(link['href'])
        numbers = string.split('=')[1]
        # check if "numbers" has instances of "+"; if so, then split the string at the "+" and list each element:
        if "+" in numbers:
            numbers = numbers.split("+")
        else:
            numbers = [numbers]
        for i in range(len(numbers)):
            numberlist.append(numbers[i])

    mostrecent=numberlist[0]
    mostrecentprefix = mostrecent[0:2]
    spec_url = 'http://basebe.obspm.fr/basebe/Spectres_png/S0'+str(mostrecentprefix)+'/sp_0'+str(mostrecent)+'.png'
    # print(spec_url)
    # Display this URL:
    display(HTML(f'<a href="{spec_url}">{spec_url}</a>'))

    response = requests.get(spec_url)
    with open(folder_name+'temp.png', 'wb') as f:
      f.write(response.content)
    fig,axs = plt.subplots(1,1,figsize=(6,6))
    img=skimage.io.imread(folder_name+'temp.png')
    axs.imshow(img)
    plt.tight_layout()
    plt.axis('off')
    plt.annotate(text=name,xy=(100,150),xytext=(100,150),color='black',size=20)
    # plt.savefig(name+'.png')
    plt.show()

    # append the link to the dataframe:
    filtered_df.iloc[j, filtered_df.columns.get_loc('Output_URL')] = spec_url

<div style="background-color: darkslategray; padding: 10px; border-radius: 5px">

### Get a 30'x30' finder chart (courtesy of Eric Jensen at Swarthmore)

</div>

In [ ]:
# add a new column to the dataframe to store the spectrum URL:
filtered_df['Finder_URL'] = ''

# URL of the page to scrape

for j in range(len(filtered_df)):
    name = filtered_df['Name'].iloc[j]
    name = name.replace(" ", "+")
    finder_url = 'https://astro.swarthmore.edu/transits/create_finding_chart.cgi?target='+str(name)+'&field_width=30&field_height=30'

    # append the link to the dataframe:
    filtered_df.iloc[j, filtered_df.columns.get_loc('Finder_URL')] = finder_url

In [ ]:
# Display the DataFrame with the added URL columns
with pd.option_context('display.max_columns', None, 'display.max_rows', None, 'display.width', 10000):
    columns_to_print = ['Name', 'RA', 'Dec', 'Vmag', 'SpType', 'Vsini', 'Nspectra', 'Comments', 'Simbad_URL', 'BeSS_URL', 'Output_URL', 'Finder_URL']
    print(filtered_df[columns_to_print].to_string(na_rep=''))

# Function to create HTML links with custom text
def make_html_link(url, label):
    if pd.notnull(url):  # Check if URL is not null
        return f'<a href="{url}" target="_blank">{label}</a>'
    return ''

# Apply the function to the URL columns with custom labels
filtered_df['Simbad_URL'] = filtered_df['Simbad_URL'].apply(lambda x: make_html_link(x, 'link'))
filtered_df['BeSS_URL'] = filtered_df['BeSS_URL'].apply(lambda x: make_html_link(x, 'link'))
filtered_df['Output_URL'] = filtered_df['Output_URL'].apply(lambda x: make_html_link(x, 'link'))
filtered_df['Finder_URL'] = filtered_df['Finder_URL'].apply(lambda x: make_html_link(x, 'link'))

# Convert the DataFrame to an HTML table with proper links
columns_to_print = ['Name', 'RA', 'Dec', 'Vmag', 'SpType', 'Vsini', 'Nspectra', 'Comments', 'Simbad_URL', 'BeSS_URL', 'Output_URL', 'Finder_URL']
html_table = filtered_df[columns_to_print].to_html(na_rep='', index=False, escape=False)

# Save the HTML table to html
with open(folder_name+"output_table.html", "w") as f:
    f.write(html_table)

# Convert the DataFrame to an HTML table with proper links
columns_to_print = ['Name', 'RA', 'Dec', 'Vmag', 'SpType', 'Vsini', 'Nspectra', 'Comments', 'Simbad_URL', 'BeSS_URL', 'Output_URL', 'Finder_URL']
# html_table with all entries centered:
html_table = css_style + filtered_df[columns_to_print].to_html(na_rep='', index=False, escape=False, justify='center')
# html_table = filtered_df[columns_to_print].to_html(na_rep='', index=False, escape=False)

# Save the HTML table to html
filename = str(folder_name)+'BeSS.'+str( Time.now().iso.split(' ')[0])+'.html'
with open(filename, "w") as f:
    f.write(html_table)
print('   -->> '+str(filename)+' created')

# Now use the ETC to get recommended exposure times in the broadband filters:

In [ ]:
# Print out a sch file:
titlestring = 'Be '+str(current_date)
end_date = current_date + timedelta(days=1)
print('title "'+str(titlestring)+'"')
print('observer jcannon@macalester.edu')
print('code mjc')
print('datestart '+str(current_date))
print('dateend '+str(end_date))
print('')

for i in range(len(filtered_df)):
    name = str(filtered_df['Name'].iloc[i])
    ra = str(filtered_df['RA'].iloc[i])
    dec = str(filtered_df['Dec'].iloc[i])
    v = float(filtered_df['Vmag'].iloc[i])
    ra = ra.replace(" ", ":")
    dec = dec.replace(" ", ":")
    # Calculate exp time in each filter:
    for filter in filter_list:
      calc = Exptime_Calculator(telescope, camera, name, filter, seeing_fwhm, exptime, v)
      sky_mean, sky_rms, adu_sec, adu_sum, snr, adu_peak, saturation_time = calc.sky_params("Default", exptime)
      exptime = np.round(saturation_time/2,3)
      if filter == 'ha':
        haexptime = exptime
      print('source '+"'"+str(name)+"'"+'  ra '+str(ra)+'  dec   '+str(dec)+'     filter '+str(filter)+'    binning 2x2 readout 0   nexp 2  exposure '+str(exptime)+'  #  Vmag = '+str(v))
    print('source '+"'"+str(name)+"'"+'  ra '+str(ra)+'  dec   '+str(dec)+'     filter lrg  binning 2x2 readout 0   nexp 3  exposure '+str(haexptime)+','+str(haexptime/2)+'   repositioning 2394,1597 #  Vmag = '+str(v))
    print('source '+"'"+str(name)+"'"+'  ra '+str(ra)+'  dec   '+str(dec)+'     filter hrg  binning 2x2 readout 0   nexp 3  exposure '+str(haexptime*2)+','+str(haexptime*4)+'  repositioning 2394,1597 #  Vmag = '+str(v))
    print('')

In [ ]:
titlestring = 'Be '+str(current_date)
end_date = current_date + timedelta(days=1)

filename = str(folder_name)+'BeSS.'+str( Time.now().iso.split(' ')[0])+'.sch'

with open((filename), 'w') as file:
    file.write(f'title "{titlestring}"\n')
    file.write('observer jcannon@macalester.edu\n')
    file.write('code mjc\n')
    file.write(f'datestart {current_date}\n')
    file.write(f'dateend {end_date}\n')
    file.write('\n')

    for i in range(len(filtered_df)):
      name = str(filtered_df['Name'].iloc[i])
      ra = str(filtered_df['RA'].iloc[i])
      dec = str(filtered_df['Dec'].iloc[i])
      v = float(filtered_df['Vmag'].iloc[i])
      ra = ra.replace(" ", ":")
      dec = dec.replace(" ", ":")
      # Calculate exp time in each filter:
      for filter in filter_list:
        calc = Exptime_Calculator(telescope, camera, name, filter, seeing_fwhm, exptime, v)
        sky_mean, sky_rms, adu_sec, adu_sum, snr, adu_peak, saturation_time = calc.sky_params("Default", exptime)
        exptime = np.round(saturation_time/2,3)
        file.write(f"source '{name}' ra  {ra}  dec   {dec}     filter {filter}    binning 2x2 readout 0   nexp 2  exposure {exptime}  #  Vmag = {v}\n")
        if filter == 'ha':
          haexptime = exptime
      file.write(f"source '{name}' ra  {ra}  dec   {dec}     filter lrg      binning 2x2 readout 0   nexp 2  exposure {haexptime},{haexptime/2}   repositioning 2394,1597 #  Vmag = {v}\n")
      file.write(f"source '{name}' ra  {ra}  dec   {dec}     filter hrg      binning 2x2 readout 0   nexp 2  exposure {haexptime*2},{haexptime*4},{haexptime*8}  repositioning 2394,1597 #  Vmag = {v}\n")
      file.write('\n')

print('      --->> sch file written to '+str(filename))

destination_file = os.path.splitext(filename)[0] + '.txt'  # Change extension to .txt
destination_file = filename + '.txt'  # Add extension .txt

# Copy the file
shutil.copy(filename, destination_file)
print('      --->>           copied to '+str(destination_file))
